# NB-04 — MLP Projector

**Goal:** Compare MLP vs Linear on the same toy alignment task.

MLPs add non-linearity — they can learn transformations a single linear layer cannot.

---

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from src.projectors.linear_projector import LinearProjector
from src.projectors.mlp_projector import MLPProjector

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

IN_DIM, HIDDEN, OUT_DIM = 1024, 2048, 4096
BATCH, SEQ = 4, 196
STEPS = 200

## 1. Shared toy task

We use a **non-linear** target mapping so the MLP has an advantage.

In [ ]:
def make_nonlinear_target(x: torch.Tensor) -> torch.Tensor:
    """Synthetic LLM targets with non-linear dependence on CLIP features."""
    w1 = torch.randn(IN_DIM, OUT_DIM, device=x.device) * 0.05
    w2 = torch.randn(IN_DIM, OUT_DIM, device=x.device) * 0.05
    h = torch.tanh(x @ w1)
    return h @ w2 + 0.1 * (x @ w1).sin()


clip_features = torch.randn(BATCH, SEQ, IN_DIM, device=DEVICE)
target_llm = make_nonlinear_target(clip_features)


def train_projector(model: nn.Module, name: str) -> list[float]:
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    losses = []
    for _ in range(STEPS):
        pred = model(clip_features)
        loss = nn.functional.mse_loss(pred, target_llm)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    print(f"{name} final loss: {losses[-1]:.4f}")
    return losses


linear_losses = train_projector(LinearProjector(IN_DIM, OUT_DIM), "Linear")
mlp_losses = train_projector(MLPProjector(IN_DIM, HIDDEN, OUT_DIM, num_layers=2), "MLP (2 layers)")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(linear_losses, label="Linear")
plt.plot(mlp_losses, label="MLP (2 layers)")
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("Linear vs MLP on Non-Linear Toy Task")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Depth ablation: 1 vs 2 vs 4 layers

In [ ]:
plt.figure(figsize=(8, 5))
for num_layers in [1, 2, 4]:
    torch.manual_seed(42)
    losses = train_projector(
        MLPProjector(IN_DIM, HIDDEN, OUT_DIM, num_layers=num_layers),
        f"MLP ({num_layers} layer{'s' if num_layers > 1 else ''})",
    )
    plt.plot(losses, label=f"{num_layers} layer(s)")

plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("MLP Depth Ablation")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Token budget reminder

Linear and MLP both keep **all patch tokens** — typically 196–257 per image.
Q-Former (NB-05) compresses to a fixed number of learnable queries.

In [ ]:
mlp = MLPProjector(IN_DIM, HIDDEN, OUT_DIM, num_layers=2)
x = torch.randn(1, 196, IN_DIM)
y = mlp(x)
print(f"MLP: {x.shape} → {y.shape}")
print("Same sequence length — every patch becomes an LLM token.")

## ✅ Checklist

- [ ] MLP beats or matches Linear on non-linear toy task
- [ ] Deeper MLP can fit faster but may overfit on small data
- [ ] `src/projectors/mlp_projector.py` is the production version

**Next:** `NB-05-qformer-deep-dive.ipynb`